In [1]:
import numpy as np
from ompl import base as ob
from ompl import geometric as og

from pydrake.all import (
    AddMultibodyPlantSceneGraph,
    Box,
    Cylinder,
    DiagramBuilder,
    InverseKinematics,
    MeshcatVisualizer,
    MeshcatVisualizerParams,
    RigidTransform,
    Role,
    RollPitchYaw,
    RotationMatrix,
    Solve,
    StartMeshcat,
    MultibodyPlant,
    SceneGraph,
    Sphere,
    Rgba,
    RobotDiagram,
    ModelInstanceIndex,
    CollisionCheckerParams,
    SceneGraphCollisionChecker,
    RobotDiagramBuilder,
    BsplineBasis,
    BsplineTrajectory,
    KinematicTrajectoryOptimization,
    MinimumDistanceLowerBoundConstraint,
)

from manipulation.meshcat_utils import PublishPositionTrajectory
from manipulation.station import MakeHardwareStation, LoadScenario, MakeRobotDiagram

In [2]:
meshcat = StartMeshcat()

INFO:drake:Meshcat listening for connections at http://localhost:7001


## IK Test

In [ ]:
meshcat.Delete()
builder = DiagramBuilder()
scenario = LoadScenario(filename="../scenarios/so101_block_welded.yaml")
station = MakeHardwareStation(scenario, meshcat=meshcat)
builder.AddSystem(station)
plant: MultibodyPlant = station.GetSubsystemByName("plant")
scene_graph: SceneGraph = station.GetSubsystemByName("scene_graph")
visualizer: MeshcatVisualizer = station.GetSubsystemByName("meshcat_visualizer(illustration)")

diagram = builder.Build()
context = diagram.CreateDefaultContext()
plant_context = plant.GetMyContextFromRoot(context)

box = plant.GetModelInstanceByName("box")
box_body = plant.GetRigidBodyByName("box_link", box)
X_WGoal = plant.EvalBodyPoseInWorld(plant_context, box_body)
meshcat.SetObject("goal", Sphere(0.02), rgba=Rgba(0.1, 0.9, 0.1, 1))
meshcat.SetTransform("goal", X_WGoal)

q0 = plant.GetPositions(plant_context)
so101 = plant.GetModelInstanceByName("so101")
gripper_frame = plant.GetFrameByName("gripper_link", so101)

ik = InverseKinematics(plant, plant_context)
ik.AddPositionConstraint(
    gripper_frame,
    [0.02, 0, -0.10],
    plant.world_frame(),
    X_WGoal.translation(),
    X_WGoal.translation(),
)
ik.AddOrientationConstraint(
    gripper_frame,
    RotationMatrix(),
    plant.world_frame(),
    X_WGoal.rotation().multiply(RotationMatrix.MakeZRotation(np.pi/2)),
    0.0,
)
ik.AddMinimumDistanceLowerBoundConstraint(1e-3, 1e-2)
prog = ik.get_mutable_prog()
q = ik.q()
prog.AddQuadraticErrorCost(np.identity(len(q)), q0, q)
prog.AddBoundingBoxConstraint(np.pi/4, np.pi/2, q[5])
prog.SetInitialGuess(q, q0)
result = Solve(ik.prog())
if result.is_success():
    print("IK success")
    print(result.GetSolution(ik.q()))
else:
    print("IK failure")

visualizer.ForcedPublish(visualizer.GetMyContextFromRoot(context))

## Trajectory Generation

### OMPL

In [4]:
class DrakeStateValidityChecker(ob.StateValidityChecker):
    def __init__(
        self, 
        si: ob.SpaceInformation,
        diagram: RobotDiagram,
        robot_model: ModelInstanceIndex,
        num_q: int,
    ):
        super().__init__(si)

        self.collision_checker_params = CollisionCheckerParams()
        self.collision_checker_params.model = diagram
        self.collision_checker_params.robot_model_instances = [robot_model]
        self.collision_checker_params.edge_step_size = 0.01
        self.collision_checker = SceneGraphCollisionChecker(self.collision_checker_params)

        self.context = self.collision_checker.MakeStandaloneModelContext()

        self.num_q = num_q
        self.influence_distance = 1e0

    def _stateToVector(self, state: ob.State) -> np.ndarray[np.float64]:
        return np.array([state[i] for i in range(self.num_q)], dtype=np.float64)

    def isValid(self, state):
        q = self._stateToVector(state)
        return self.collision_checker.CheckContextConfigCollisionFree(
            self.context, 
            q
        )
    
    def clearance(self, state):
        q = self._stateToVector(state)
        distances = self.collision_checker.CalcContextRobotClearance(
            self.context,
            q,
            self.influence_distance,
        ).distances()
        if distances.size > 0:
            return np.min(distances.distances())
        else:
            return np.inf


def GeneratePathOMPL(
    plant: MultibodyPlant, 
    diagram: RobotDiagram,
    robot_model: ModelInstanceIndex,
    q_start: np.ndarray, 
    q_goal: np.ndarray,
    solve_duration = 1.0,
) -> np.ndarray | None:    
    num_q = plant.num_positions()
    lower_bounds = plant.GetPositionLowerLimits()
    upper_bounds = plant.GetPositionUpperLimits()
    print(num_q)

    bounds = ob.RealVectorBounds(num_q)
    for i in range(num_q):
        bounds.setLow(i, lower_bounds[i])
        bounds.setHigh(i, upper_bounds[i])

    space = ob.RealVectorStateSpace(num_q)
    space.setBounds(bounds)
    
    si = ob.SpaceInformation(space)
    validityChecker = DrakeStateValidityChecker(si, diagram, robot_model, num_q)
    si.setStateValidityChecker(validityChecker)
    si.setup()
    print(si)

    start = space.allocState()
    goal = space.allocState()
    for i in range(num_q):
        start[i] = q_start[i]
        goal[i] = q_goal[i]

    pdef = ob.ProblemDefinition(si)
    pdef.setStartAndGoalStates(start, goal)

    objective = ob.PathLengthOptimizationObjective(si)
    pdef.setOptimizationObjective(objective)
    print(pdef)

    # planner = og.RRTstar(si)
    planner = og.RRTConnect(si, addIntermediateStates=True)
    # planner = og.RRT(si)
    # planner = og.PRM(si)
    planner.setProblemDefinition(pdef)
    planner.setup()

    solved = planner.solve(solve_duration)
    print(solved.asString())
    if solved:
        path = pdef.getSolutionPath()
        num_states = path.getStateCount()
        dim = space.getDimension()

        path_array = np.zeros((num_states, dim))
        for i in range(num_states):
            state = path.getState(i)
            for j in range(dim):
                path_array[i, j] = state[j]

        if np.any(q_start != path_array[0]) or np.any(q_goal != path_array[-1]):
            return None
        
        print(path_array)
        if path_array.shape[0] < 4:
            path_array = np.vstack([
                np.linspace(path_array[0], path_array[1], 6 - path_array.shape[0]),
                path_array[2:]
            ])

        waypoints = path_array.T
        return waypoints
    else:
        return None

### Trajectory Optimization

In [5]:
def GenerateTrajectory(
    plant: MultibodyPlant,
    diagram: RobotDiagram,
    robot_model: ModelInstanceIndex,
    waypoints: np.ndarray,
    avoid_collisions = True,
) -> BsplineTrajectory | None:
    num_q = plant.num_positions()
    basis = BsplineBasis(4, waypoints.shape[1])
    init_traj = BsplineTrajectory(basis, waypoints)
    trajopt = KinematicTrajectoryOptimization(num_q, waypoints.shape[1], 4)
    trajopt.SetInitialGuess(init_traj)

    trajopt.AddDurationCost(1.0)
    trajopt.AddPathLengthCost(1.0)
    trajopt.AddPositionBounds(
        plant.GetPositionLowerLimits(), plant.GetPositionUpperLimits()
    )
    trajopt.AddVelocityBounds(
        plant.GetVelocityLowerLimits(), plant.GetVelocityUpperLimits()
    )
    trajopt.AddDurationConstraint(0.5, 5.0)

    q_start = waypoints[:, 0]
    q_goal = waypoints[:, -1]
    trajopt.AddPathPositionConstraint(lb=q_start, ub=q_start, s=0)
    trajopt.AddPathPositionConstraint(lb=q_goal, ub=q_goal, s=1)

    trajopt.AddPathVelocityConstraint(np.zeros((num_q, 1)), np.zeros((num_q, 1)), 0)
    trajopt.AddPathVelocityConstraint(np.zeros((num_q, 1)), np.zeros((num_q, 1)), 1)

    if avoid_collisions:
        collision_checker_params = CollisionCheckerParams()
        collision_checker_params.model = diagram
        collision_checker_params.robot_model_instances = [robot_model]
        collision_checker_params.edge_step_size = 0.01
        collision_checker = SceneGraphCollisionChecker(collision_checker_params)
        collision_checker.SetPaddingAllRobotEnvironmentPairs(1e-3)
        collision_constraint = MinimumDistanceLowerBoundConstraint(
            collision_checker,
            1e-3,
            collision_checker.MakeStandaloneModelContext(),
            None,
            1e-2,
        )
        evaluate_at_s = np.linspace(0, 1, 25)
        for s in evaluate_at_s:
            trajopt.AddPathPositionConstraint(collision_constraint, s)

    prog = trajopt.get_mutable_prog()
    result = Solve(prog)
    if result.is_success():
        return trajopt.ReconstructTrajectory(result)
    else:
        return None

### Inverse Kinematics

In [6]:
def GenerateGoalConfig(
    plant: MultibodyPlant,
    diagram: RobotDiagram,
    robot_model: ModelInstanceIndex,
    q0: np.ndarray,
) -> np.ndarray | None:
    context = diagram.CreateDefaultContext()
    plant_context = plant.GetMyContextFromRoot(context)

    box = plant.GetModelInstanceByName("box")
    box_body = plant.GetRigidBodyByName("box_link", box)
    X_WGoal = plant.EvalBodyPoseInWorld(plant_context, box_body)

    gripper_frame = plant.GetFrameByName("gripper_link", robot_model)

    ik = InverseKinematics(plant, plant_context)
    ik.AddPositionConstraint(
        gripper_frame,
        [0.015, 0, -0.10],
        plant.world_frame(),
        X_WGoal.translation(),
        X_WGoal.translation(),
    )
    ik.AddOrientationConstraint(
        gripper_frame,
        RotationMatrix(),
        plant.world_frame(),
        X_WGoal.rotation().multiply(RotationMatrix.MakeZRotation(np.pi/2)),
        0.0,
    )
    ik.AddMinimumDistanceLowerBoundConstraint(5e-3, 1e-2)
    prog = ik.get_mutable_prog()
    q = ik.q()
    prog.AddQuadraticErrorCost(np.identity(len(q)), q0, q)
    prog.AddBoundingBoxConstraint(np.pi/4, np.pi/2, q[5])
    prog.SetInitialGuess(q, q0)
    result = Solve(ik.prog())
    if result.is_success():
        return result.GetSolution(ik.q())
    else:
        return None

### Full Solution

In [8]:
meshcat.Delete()
builder = RobotDiagramBuilder()
plant = builder.plant()
scene_graph = builder.scene_graph()
parser = builder.parser()

mat = parser.AddModelsFromUrl("file:///home/noor/so101-drake/models/objects/mat.sdf")[0]
plant.WeldFrames(plant.world_frame(), plant.GetFrameByName("mat_link"))
so101 = parser.AddModelsFromUrl("file:///home/noor/so101-drake/models/SO101/so101_new_calib_drake.urdf")[0]
plant.WeldFrames(
    plant.GetFrameByName("mat_link"),
    plant.GetFrameByName("base_link"), 
    RigidTransform(
        RotationMatrix.MakeZRotation(np.pi / 2),
        [0, -0.1775, 0.0074]
    )
)
box = parser.AddModelsFromUrl("file:///home/noor/so101-drake/models/objects/box.sdf")[0]
plant.WeldFrames(
    plant.GetFrameByName("mat_link"),
    plant.GetFrameByName("box_link"),
    RigidTransform(
        RotationMatrix.MakeZRotation(np.pi / 2),
        # [-0.05, 0.05, 0.02],
        [-0.075, 0.025, 0.02],
    )
)

plant.Finalize()

q_rest = np.array([0, -1.822, 1.55, 0.906, 0, 0.0])
plant.SetDefaultPositions(so101, q_rest)

visualizer = MeshcatVisualizer.AddToBuilder(
    builder.builder(),
    scene_graph,
    meshcat,
    MeshcatVisualizerParams(role=Role.kIllustration),
)
collision_visualizer = MeshcatVisualizer.AddToBuilder(
    builder.builder(),
    scene_graph,
    meshcat,
    MeshcatVisualizerParams(
        prefix="collision", role=Role.kProximity, visible_by_default=False
    ),
)

diagram: RobotDiagram = builder.Build()

context = diagram.CreateDefaultContext()
plant_context = plant.GetMyContextFromRoot(context)
q_start = plant.GetPositions(plant_context)
q_goal = GenerateGoalConfig(plant, diagram, so101, q_start)

if q_goal is not None:
    print("IK success")
    print(q_start)
    print(q_goal)

    waypoints = GeneratePathOMPL(
        plant=plant,
        diagram=diagram,
        robot_model=so101,
        q_start=q_start,
        q_goal=q_goal,
        solve_duration=10.0,
    )

    if waypoints is not None:
        print("OMPL success")
        print(waypoints)

        trajectory = GenerateTrajectory(
            plant=plant, 
            diagram=diagram, 
            robot_model=so101, 
            waypoints=waypoints, 
            avoid_collisions=False
        )

        if trajectory is not None:
            print("Trajectory optimization success")

            meshcat.Flush()
            context = diagram.CreateDefaultContext()
            PublishPositionTrajectory(trajectory, context, plant, visualizer)
            collision_visualizer.ForcedPublish(collision_visualizer.GetMyContextFromRoot(context))
        
        else:
            print("Trajectory optimization failure")
    
    else:
        print("OMPL failure")

else:
    print("IK failure")

IK failure
